In [17]:
!pip install ipywidgets

  Obtaining dependency information for ipywidgets from https://files.pythonhosted.org/packages/56/6d/0d9848617b9f753b87f214f1c682592f7ca42de085f564352f10f0843026/ipywidgets-8.1.8-py3-none-any.whl.metadata
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Predictions of an LLM
We first just take a simple model and load into memory, and then see what the output for the next token is.

We calculate the entropy, as well as observe the distribution for the top 10 tokens.

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt")

torch.manual_seed(42)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, -1, :]
probs = torch.softmax(logits, dim=-1)
entropy = -(probs * torch.log2(probs)).sum()
print(entropy.item())

topk = torch.topk(probs, 10)

for p, idx in zip(topk.values[0], topk.indices[0]):
    token = tokenizer.decode([idx.item()])
    prob = p.item()

    print(f"{repr(token):20} {prob:.4f}")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7331.20it/s]


4.5
' Paris'             0.2949
' ______'            0.1230
':\n'                0.0659
':\n\n'              0.0547
' located'           0.0513
' __'                0.0483
' ____'              0.0400
' the'               0.0376
' ('                 0.0259
' ['                 0.0228


# Growing certainty of predictions over time
We want to now give the LLM a small piece of text, and ask it to predict each next token in turn.

We are curious to see how the entropy for each token changes over time.

We give it a nonsense piece of text (written by me), and a piece of text reciting a famous piece of history.
Logically we expect the nonsense to be much more information dense, due to the random nature of the phrase, than the logical recitation of a well known piece of history.

In [19]:
sekigahara = """The battle of Sekigahara was fought in the plain of Sekigahara, nestled in the mountains east of Lake Biwa in central Japan.
Largely regarded as the definitive battle in the Sengoku Jidai, it was fought between supporters of Toyotomi
Hideyori, the son of Toyotomi Hideyoshi, and Tokugawa Ieyasu. The battle was a decisive victory for the
Tokugawa forces, leading to the unification of Japan and the establishment of the Tokugawa Shogunate."""

nonsense = """Blue cats are red. Following this cereal rained from the earth. Gained was everything.
Though longing for custard, television was discarded. How scissors take pens to create pepper. Totyota
are mighty ducks. Apple Jesus ran under Jupiter. Cards eaten ooze flowers within the sky. Shoes drank your
second ninja tape. Conflagration of fluffy diamonds sapped eye forks. Gratuitous toes experience Michelin temples."""

In [20]:
def predictions_over_time(text, model, tokenizer):
    tokens = tokenizer(text, return_tensors="pt")["input_ids"][0]

    torch.manual_seed(42)
    total_entropy = 0
    with torch.inference_mode():
        for i in range(len(tokens) - 1):
            # Context seen so far
            context = tokens[:i+1].unsqueeze(0)

            outputs = model(context)
            logits = outputs.logits[0, -1]
            probs = torch.softmax(logits, dim = 0)

            actual_next = tokens[i+1]
            actual_prob = probs[actual_next].item()
            predicted = probs.argmax().item()
            entropy = -(probs * torch.log2(probs)).sum().item()
            total_entropy += entropy

            print("="*40)
            print("Context   : ", repr(tokenizer.decode(context[0])))
            print("Actual    :", repr(tokenizer.decode([actual_next])))
            print("Predicted :", repr(tokenizer.decode([predicted])))
            print("P(actual) :", actual_prob)
            print("Entropy   :", entropy)
            print("Total Etropy :", total_entropy)


In [21]:
predictions_over_time(sekigahara, model, tokenizer)

Context   :  'The'
Actual    : ' battle'
Predicted : ' following'
P(actual) : 1.519918441772461e-05
Entropy   : 7.59375
Total Etropy : 7.59375
Context   :  'The battle'
Actual    : ' of'
Predicted : ' of'
P(actual) : 0.30078125
Entropy   : 5.0
Total Etropy : 12.59375
Context   :  'The battle of'
Actual    : ' Sek'
Predicted : ' the'
P(actual) : 0.000652313232421875
Entropy   : 8.75
Total Etropy : 21.34375
Context   :  'The battle of Sek'
Actual    : 'ig'
Predicted : 'ond'
P(actual) : 0.03662109375
Entropy   : 6.9375
Total Etropy : 28.28125
Context   :  'The battle of Sekig'
Actual    : 'ah'
Predicted : 'ah'
P(actual) : 0.77734375
Entropy   : 1.46875
Total Etropy : 29.75
Context   :  'The battle of Sekigah'
Actual    : 'ara'
Predicted : 'ara'
P(actual) : 0.98828125
Entropy   : 0.1259765625
Total Etropy : 29.8759765625
Context   :  'The battle of Sekigahara'
Actual    : ' was'
Predicted : ' was'
P(actual) : 0.26171875
Entropy   : 4.65625
Total Etropy : 34.5322265625
Context   :  'The bat

In [22]:
predictions_over_time(nonsense, model, tokenizer)

Context   :  'Blue'
Actual    : ' cats'
Predicted : '-F'
P(actual) : 0.00051116943359375
Entropy   : 10.5
Total Etropy : 10.5
Context   :  'Blue cats'
Actual    : ' are'
Predicted : ' are'
P(actual) : 0.294921875
Entropy   : 4.65625
Total Etropy : 15.15625
Context   :  'Blue cats are'
Actual    : ' red'
Predicted : ' more'
P(actual) : 0.000499725341796875
Entropy   : 7.6875
Total Etropy : 22.84375
Context   :  'Blue cats are red'
Actual    : '.'
Predicted : ' cats'
P(actual) : 0.08935546875
Entropy   : 3.5625
Total Etropy : 26.40625
Context   :  'Blue cats are red.'
Actual    : ' Following'
Predicted : ' If'
P(actual) : 5.602836608886719e-05
Entropy   : 6.5625
Total Etropy : 32.96875
Context   :  'Blue cats are red. Following'
Actual    : ' this'
Predicted : ' this'
P(actual) : 0.55859375
Entropy   : 2.8125
Total Etropy : 35.78125
Context   :  'Blue cats are red. Following this'
Actual    : ' cereal'
Predicted : ' rule'
P(actual) : 3.655441105365753e-08
Entropy   : 3.515625
Total Etrop